# 03 - Feature Engineering

**Kenya Food Price Inflation Tracker**

## Objective
Create features for time series forecasting models.

## Features to Create
1. Lag features (1, 3, 6, 12 months)
2. Rolling statistics (3, 6, 12 month windows)
3. Time-based features (month, quarter, year)
4. Inflation rates (MoM, YoY)
5. Seasonal indicators

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# Load monthly aggregated data
try:
    df = pd.read_csv('../data/clean/wfp_monthly_avg.csv', parse_dates=['date'])
except FileNotFoundError:
    df = pd.read_csv('data/clean/wfp_monthly_avg.csv', parse_dates=['date'])

print(f"Dataset shape: {df.shape}")
df.head()


Dataset shape: (4160, 9)


,date,commodity,region,market,price_mean,price_std,obs_count,market_id,currency
0,2006-01-01,Beans (dry) - Retail,Eastern,Kitui,39.000,NaN,1,187,KES
1,2006-01-01,Beans - Wholesale,Coast,Mombasa,33.630,NaN,1,191,KES
2,2006-01-01,Beans - Wholesale,Nairobi,Nairobi,42.311,NaN,1,184,KES
3,2006-01-01,Beans - Wholesale,Nyanza,Kisumu,39.608,NaN,1,186,KES
4,2006-01-01,Beans - Wholesale,Rift Valley,Eldoret town,45.854,NaN,1,185,KES


## 1. Focus on Maize (White) - Retail for Modeling

In [3]:
# Filter to Maize retail prices, aggregate to national average
maize = df[df['commodity'] == 'Maize (white) - Retail'].copy()
maize_national = maize.groupby('date').agg({
    'price_mean': 'mean',
    'obs_count': 'sum'
}).reset_index()
maize_national.columns = ['date', 'price', 'observations']
maize_national = maize_national.sort_values('date').reset_index(drop=True)

print(f"Maize national dataset: {maize_national.shape}")
print(f"Date range: {maize_national['date'].min()} to {maize_national['date'].max()}")
maize_national.head(10)

Maize national dataset: (180, 3)
Date range: 2006-01-01 00:00:00 to 2020-12-01 00:00:00


,date,price,observations
0,2006-01-01,23.50,4
1,2006-02-01,24.25,4
2,2006-03-01,23.50,4
3,2006-04-01,25.25,4
4,2006-05-01,23.75,4
5,2006-06-01,22.75,4
6,2006-07-01,23.00,4
7,2006-08-01,23.25,4
8,2006-09-01,22.50,4
9,2006-10-01,23.25,4


## 2. Create Time-Based Features

In [4]:
# Extract time components
maize_national['year'] = maize_national['date'].dt.year
maize_national['month'] = maize_national['date'].dt.month
maize_national['quarter'] = maize_national['date'].dt.quarter
maize_national['day_of_year'] = maize_national['date'].dt.dayofyear

# Cyclical encoding for month
maize_national['month_sin'] = np.sin(2 * np.pi * maize_national['month'] / 12)
maize_national['month_cos'] = np.cos(2 * np.pi * maize_national['month'] / 12)

print("✓ Time-based features created")
maize_national[['date', 'price', 'year', 'month', 'quarter', 'month_sin', 'month_cos']].head()

✓ Time-based features created


,date,price,year,month,quarter,month_sin,month_cos
0,2006-01-01,23.50,2006,1,1,0.500000,8.660254e-01
1,2006-02-01,24.25,2006,2,1,0.866025,5.000000e-01
2,2006-03-01,23.50,2006,3,1,1.000000,6.123234e-17
3,2006-04-01,25.25,2006,4,2,0.866025,-5.000000e-01
4,2006-05-01,23.75,2006,5,2,0.500000,-8.660254e-01


## 3. Create Lag Features

In [5]:
# Create lag features
for lag in [1, 3, 6, 12]:
    maize_national[f'price_lag_{lag}m'] = maize_national['price'].shift(lag)

print("✓ Lag features created")
cols = ['date', 'price'] + [f'price_lag_{i}m' for i in [1, 3, 6, 12]]
maize_national[cols].head(15)

✓ Lag features created


,date,price,price_lag_1m,price_lag_3m,price_lag_6m,price_lag_12m
0,2006-01-01,23.50,NaN,NaN,NaN,NaN
1,2006-02-01,24.25,23.50,NaN,NaN,NaN
2,2006-03-01,23.50,24.25,NaN,NaN,NaN
3,2006-04-01,25.25,23.50,23.50,NaN,NaN
4,2006-05-01,23.75,25.25,24.25,NaN,NaN
5,2006-06-01,22.75,23.75,23.50,NaN,NaN
6,2006-07-01,23.00,22.75,25.25,23.50,NaN
7,2006-08-01,23.25,23.00,23.75,24.25,NaN
8,2006-09-01,22.50,23.25,22.75,23.50,NaN
9,2006-10-01,23.25,22.50,23.00,25.25,NaN


## 4. Create Rolling Statistics

In [6]:
# Rolling mean and std for different windows
for window in [3, 6, 12]:
    maize_national[f'price_rolling_mean_{window}m'] = maize_national['price'].rolling(window=window).mean()
    maize_national[f'price_rolling_std_{window}m'] = maize_national['price'].rolling(window=window).std()

print("✓ Rolling statistics created")
cols = ['date', 'price', 'price_rolling_mean_3m', 'price_rolling_std_3m', 'price_rolling_mean_12m']
maize_national[cols].head(15)

✓ Rolling statistics created


,date,price,price_rolling_mean_3m,price_rolling_std_3m,price_rolling_mean_12m
0,2006-01-01,23.50,NaN,NaN,NaN
1,2006-02-01,24.25,NaN,NaN,NaN
2,2006-03-01,23.50,23.750000,0.433013,NaN
3,2006-04-01,25.25,24.333333,0.877971,NaN
4,2006-05-01,23.75,24.166667,0.946485,NaN
5,2006-06-01,22.75,23.916667,1.258306,NaN
6,2006-07-01,23.00,23.166667,0.520416,NaN
7,2006-08-01,23.25,23.000000,0.250000,NaN
8,2006-09-01,22.50,22.916667,0.381881,NaN
9,2006-10-01,23.25,23.000000,0.433013,NaN


## 5. Create Inflation Rates

In [7]:
# Month-over-Month (MoM) inflation
maize_national['inflation_mom'] = ((maize_national['price'] - maize_national['price_lag_1m']) / maize_national['price_lag_1m']) * 100

# Year-over-Year (YoY) inflation
maize_national['inflation_yoy'] = ((maize_national['price'] - maize_national['price_lag_12m']) / maize_national['price_lag_12m']) * 100

print("✓ Inflation rates calculated")
maize_national[['date', 'price', 'inflation_mom', 'inflation_yoy']].tail(20)

✓ Inflation rates calculated


,date,price,inflation_mom,inflation_yoy
160,2019-05-01,55.900000,5.893496,6.883365
161,2019-06-01,55.444444,-0.814947,6.737968
162,2019-07-01,57.555556,3.807615,14.046675
163,2019-08-01,58.455556,1.563707,25.891362
164,2019-09-01,56.166667,-3.915605,21.048851
165,2019-10-01,54.388889,-3.165183,12.616758
166,2019-11-01,55.788889,2.574055,20.697115
167,2019-12-01,56.433333,1.155148,19.674835
168,2020-01-01,54.650000,-3.160071,5.070896
169,2020-02-01,53.833333,-1.494358,2.320424


## 6. Create Seasonal Indicators

In [8]:
# Kenya's maize seasons:
# Harvest seasons: April-May (long rains), Oct-Nov (short rains)
# Lean season: Jan-Mar

def get_season(month):
    if month in [4, 5]:
        return 'harvest_long'
    elif month in [10, 11]:
        return 'harvest_short'
    elif month in [1, 2, 3]:
        return 'lean'
    else:
        return 'normal'

maize_national['season'] = maize_national['month'].apply(get_season)

# One-hot encode seasons
season_dummies = pd.get_dummies(maize_national['season'], prefix='season')
maize_national = pd.concat([maize_national, season_dummies], axis=1)

print("✓ Seasonal indicators created")
print(maize_national['season'].value_counts())

✓ Seasonal indicators created
season
normal           75
lean             45
harvest_long     30
harvest_short    30
Name: count, dtype: int64


## 7. Save Engineered Dataset

In [9]:
# Save full feature-engineered dataset
maize_national.to_csv('../data/clean/maize_features.csv', index=False)
print(f"✓ Saved: maize_features.csv ({len(maize_national)} rows, {len(maize_national.columns)} columns)")

# Also save a version without NaN (for ML models)
maize_ml = maize_national.dropna()
maize_ml.to_csv('../data/clean/maize_features_ml.csv', index=False)
print(f"✓ Saved: maize_features_ml.csv ({len(maize_ml)} rows, {len(maize_ml.columns)} columns)")

print("\n=== Feature Engineering Summary ===")
print(f"Total features: {len(maize_national.columns)}")
print(f"Rows with complete data: {len(maize_ml)}")
print(f"Date range (ML): {maize_ml['date'].min()} to {maize_ml['date'].max()}")

print("\n✓ Feature engineering completed!")

✓ Saved: maize_features.csv (180 rows, 26 columns)
✓ Saved: maize_features_ml.csv (168 rows, 26 columns)

=== Feature Engineering Summary ===
Total features: 26
Rows with complete data: 168
Date range (ML): 2007-01-01 00:00:00 to 2020-12-01 00:00:00

✓ Feature engineering completed!
